<a href="https://colab.research.google.com/github/Dhanshree010/pattern-recognition-practical/blob/main/practical_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [4]:
df = pd.read_csv("knn.csv")
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
0,LP001015,Male,Yes,0,Graduate,No,5720,0,110.0,360.0,1.0,Urban
1,LP001022,Male,Yes,1,Graduate,No,3076,1500,126.0,360.0,1.0,Urban
2,LP001031,Male,Yes,2,Graduate,No,5000,1800,208.0,360.0,1.0,Urban
3,LP001035,Male,Yes,2,Graduate,No,2340,2546,100.0,360.0,NaN,Urban
4,LP001051,Male,No,0,Not Graduate,No,3276,0,78.0,360.0,1.0,Urban


In [5]:
df.info()

df.isnull().sum()

df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 367 entries, 0 to 366
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            367 non-null    object 
 1   Gender             356 non-null    object 
 2   Married            367 non-null    object 
 3   Dependents         357 non-null    object 
 4   Education          367 non-null    object 
 5   Self_Employed      344 non-null    object 
 6   ApplicantIncome    367 non-null    int64  
 7   CoapplicantIncome  367 non-null    int64  
 8   LoanAmount         362 non-null    float64
 9   Loan_Amount_Term   361 non-null    float64
 10  Credit_History     338 non-null    float64
 11  Property_Area      367 non-null    object 
dtypes: float64(3), int64(2), object(7)
memory usage: 34.5+ KB


(367, 12)

In [6]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

cat_cols = df.select_dtypes(include=["object"]).columns

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [7]:
df = df.drop("Loan_ID", axis=1)

In [8]:
encoder = LabelEncoder()

for col in df.select_dtypes(include="object").columns:
    df[col] = encoder.fit_transform(df[col])

df.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
0,1,1,0,0,0,5720,0,110.0,360.0,1.0,2
1,1,1,1,0,0,3076,1500,126.0,360.0,1.0,2
2,1,1,2,0,0,5000,1800,208.0,360.0,1.0,2
3,1,1,2,0,0,2340,2546,100.0,360.0,1.0,2
4,1,0,0,1,0,3276,0,78.0,360.0,1.0,2


In [13]:
print(df.columns)

Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area'],
      dtype='object')


In [15]:
df["Loan_Status"] = np.where(
    (df["Credit_History"] == 1) & (df["ApplicantIncome"] > 3000),
    1,
    0
)

In [17]:
X = df.drop("Loan_Status", axis=1)

y = df["Loan_Status"]

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [19]:
dt_model = DecisionTreeClassifier(
    criterion="entropy",
    random_state=42
)

dt_model.fit(X_train, y_train)

DecisionTreeClassifier(criterion='entropy', random_state=42)

In [20]:
dt_prediction = dt_model.predict(X_test)

In [21]:
print("Accuracy:", accuracy_score(y_test, dt_prediction))

print(confusion_matrix(y_test, dt_prediction))

print(classification_report(y_test, dt_prediction))

Accuracy: 1.0
[[37  0]
 [ 0 37]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        37
           1       1.00      1.00      1.00        37

    accuracy                           1.00        74
   macro avg       1.00      1.00      1.00        74
weighted avg       1.00      1.00      1.00        74



In [22]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [23]:
knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model.fit(X_train_scaled, y_train)

KNeighborsClassifier()

In [24]:
knn_prediction = knn_model.predict(X_test_scaled)

In [25]:
print("Accuracy:", accuracy_score(y_test, knn_prediction))

print(confusion_matrix(y_test, knn_prediction))

print(classification_report(y_test, knn_prediction))

Accuracy: 0.7702702702702703
[[24 13]
 [ 4 33]]
              precision    recall  f1-score   support

           0       0.86      0.65      0.74        37
           1       0.72      0.89      0.80        37

    accuracy                           0.77        74
   macro avg       0.79      0.77      0.77        74
weighted avg       0.79      0.77      0.77        74



In [26]:
results = pd.DataFrame(
    {
        "Model": ["Decision Tree", "KNN"],
        "Accuracy": [
            accuracy_score(y_test, dt_prediction),
            accuracy_score(y_test, knn_prediction),
        ],
    }
)

print(results)

           Model  Accuracy
0  Decision Tree   1.00000
1            KNN   0.77027


In [27]:
importance = pd.DataFrame(
    {
        "Feature": X.columns,
        "Importance": dt_model.feature_importances_,
    }
)

importance = importance.sort_values(
    by="Importance",
    ascending=False,
)

print(importance)

              Feature  Importance
5     ApplicantIncome    0.632954
9      Credit_History    0.367046
0              Gender    0.000000
2          Dependents    0.000000
1             Married    0.000000
4       Self_Employed    0.000000
3           Education    0.000000
6   CoapplicantIncome    0.000000
7          LoanAmount    0.000000
8    Loan_Amount_Term    0.000000
10      Property_Area    0.000000
